In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        

import kagglehub

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/aliiihussain/car-price-prediction/car_price_prediction_.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
num_cols = df.select_dtypes(include = 'number').columns
num_cols

In [ ]:
for col in num_cols:
    plt.figure(figsize = (6,3))
    sns.histplot(df[col] , kde = True)
    plt.show()

In [ ]:
for col in num_cols:
    plt.figure(figsize = (6,3))
    sns.boxplot(df[col])
    plt.show()

In [ ]:
sns.heatmap(df[num_cols].corr() , annot = True )

In [ ]:
df.head()

In [ ]:
df_edited = df.copy()

df_edited['Transmission'] = df_edited['Transmission'].str.strip()
df_edited['Transmission'] = df_edited['Transmission'].map({
    "Manual": 0,
    "Automatic": 1
})

In [ ]:
df_edited = df_edited.drop(columns = ["Car ID"])

In [ ]:
df_edited.head()

In [ ]:
df_edited['Condition'].value_counts()
df_edited = pd.get_dummies(df_edited , columns = ['Fuel Type' , 'Condition'] , drop_first = True)

In [ ]:
df_edited.head()

In [ ]:
mode_avg = df_edited.groupby('Model')['Price'].mean().round(2)
df['model_encoded'] = df_edited['Model'].map(mode_avg)
brand_avg = df_edited.groupby('Brand')['Price'].mean().round(2)
df['brand_encoded'] = df_edited['Brand'].map(brand_avg)
df_edited = df_edited.drop(columns = ['Model' , 'Brand'])

In [ ]:
df_edited = df_edited.astype(int)

In [ ]:
df_edited['car_age'] = 2026 - df_edited['Year']
df_edited = df_edited.drop(columns = ['Year'])

In [ ]:
df_edited['Engine Size'] = df['Engine Size']

In [ ]:
df_edited.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = ['Engine Size' , 'Mileage' , 'model_encoded' , 'brand_encoded']

df_edited[num_cols] = scaler.fit_transform(df_edited[num_cols])

df_edited.head()

In [ ]:
from scipy.stats import pearsonr

num_cols = df_edited.select_dtypes(include = 'number').drop(columns = 'Price').columns

correlation = {
    feature : pearsonr(df_edited[feature] , df_edited['Price'])[0]
    for feature in num_cols
}

correlation_df = pd.DataFrame(list(correlation.items()) , columns = ['Feature' , 'Pearsonr_relation'])
correlation_df.sort_values('Pearsonr_relation' , ascending = False)

In [ ]:
from scipy.stats import chi2_contingency
category_cols = ['Fuel Type_Electric','Fuel Type_Hybrid','Fuel Type_Petrol','Condition_New','Condition_Used','Transmission']
df_edited['Price_category'] = pd.qcut(df_edited['Price'] , q = 4)
chi2_results = {}
alpha = 0.05
for col in category_cols:
    contegency = pd.crosstab(df_edited[col] , df_edited['Price_category'])
    chi2_stat , p_val, _ , _ = chi2_contingency(contegency)
    desicion = 'Accept (keep Feature)' if p_val < alpha else 'Reject (Drop Feature)'
    chi2_results[col] = {
        'chi2stat' : chi2_stat,
        'p_val' : p_val,
        'desicion' : desicion
    }

result_df = pd.DataFrame(chi2_results).T
result_df.sort_values('p_val' , ascending = False)


Data is artificial so all data almost unreasonable still creating model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
x = df_edited.drop(columns = ['Price' , 'Price_category'])
y = df_edited['Price']
train_x , test_x , train_y , test_y = train_test_split(x , y, test_size = 0.2 , random_state = 42)
linear = LinearRegression()

linear.fit(train_x , train_y)
predict_y = linear.predict(test_x)

r2 = r2_score(test_y , predict_y)
r2

Model Evaluation
A Linear Regression model was trained on the processed feature space using an 80/20 train-test split. 

* **Empirical Result:** R^2 = 0.011991512965676243
* **Interpretation:** The model can only explain 1.19 % of the variance in car prices. 

This near-zero R^2 score empirically validates the prior Pearson Correlation and Chi-Square (p > 0.05) audits.
It provides mathematical proof that the dataset is completely decoupled from the target variable (synthetic noise), demonstrating that the data pipeline is structurally flawless but working with zero underlying statistical signal.